# 03 – Post-process 2D output

Equivalent to `post_pro_rel_2D.ipynb`.

In [ ]:
cd(@__DIR__)
include(joinpath("..","src","PIC.jl"))
using .PIC
using Plots
using Statistics
using FFTW
using FileIO, JLD2


## Choose output file and load metadata

In [ ]:
out_file = joinpath("..", "Results",
    "thermal_rel_newmodule_J50x50_N5_Th3_alp1_o5.jld2")

data = load(out_file)
N, J, box, order = data["par_grid"]
t_i, t_f, M, M_g, dt = data["par_evolv"]
g = PICGrid(box, J, order)

t_series = range(t_i, t_f, length=M_g)
dx = differentials(g)
x_p = [dx[1] * (i - 1) for i in 1:J[1]]
y_p = [dx[2] * (i - 1) for i in 1:J[2]]

@show data["run_name"]


## Load averaged fields

In [ ]:
avg = load_averages(out_file, g, M_g;
    fields=(:n, :S, :E, :B, :Energy_K, :Energy_E, :T))

n_t   = avg[:n]      # (J1,J2,M_g)
S_t   = avg[:S]      # (2,J1,J2,M_g)
E_t   = avg[:E]      # (2,J1,J2,M_g)
B_t   = avg[:B]      # (J1,J2,M_g)
Energy_K = avg[:Energy_K][:]  # scalar per output
Energy_E = avg[:Energy_E][:]
T_t   = avg[:T][:]


## Energy history

In [ ]:
plot(t_series, Energy_E, label="E_E", xlabel="t", ylabel="energy")
plot!(t_series, Energy_K, label="E_K")
plot!(t_series, Energy_E .+ Energy_K, label="E_total")


## Density spectrum at final time

In [ ]:
rho_f = rfft(n_t[:, :, M_g] .- 1.0)
freqs = rfftfreq(J[1], 1 / dx[1])
Plots.scatter(freqs, abs.(rho_f[:, 1]),
    title="density spectrum", xlabel="k_x", ylabel="|ρ̂|",
    xlim=(0, 6), label=false)


## Final density and fields

In [ ]:
p1 = heatmap(x_p, y_p, n_t[:, :, M_g]', title="n", aspectratio=1)
p2 = heatmap(x_p, y_p, E_t[1, :, :, M_g]', title="E1", aspectratio=1)
p3 = heatmap(x_p, y_p, B_t[:, :, M_g]', title="B", aspectratio=1)
plot(p1, p2, p3, layout=(1, 3), size=(900, 250))


## Temperature history

In [ ]:
plot(t_series, T_t, xlabel="t", ylabel="T", title="temperature", label=false)
